# Data Query & Save

This script saves data of annotation tables, functional data, and skeleton tables

<b>Warning: The project folder has changed; REWRITE THE CODE WITH UPDATED PATH PLZ<b>

## Setup

In [6]:
import numpy as np
import pandas as pd
import pickle
import math
from collections import deque
import os
import glob
import sys
import tqdm
import caveclient

from meshparty import skeleton
import skeleton_plot
from meshparty import trimesh_io
import meshparty

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import plotly.graph_objects as go
sys.path.append(os.path.abspath('./analysis'))

# Import dendrite libraries
import dendrites as dd
import skeleton_plots as skp
import synapse_analysis as synapse_anal

# Caveclient setup
client = caveclient.CAVEclient('minnie65_public')
client.version = 1300

In [7]:
# Path
path_project = "/media/DATA1/CK/python_script/MicronsBinder_04"

## Annotation tables

Query annotation tables from caveclient\
You can either query them, or read them from your local files (version=1412)

In [8]:
# ====== Read from cavecleint =========
coregistration_manual_v4 = client.materialize.query_table('coregistration_manual_v4')
nucleus_detection_v0= client.materialize.query_table('nucleus_detection_v0')
aibs_metamodel_celltypes_v661= client.materialize.query_table('aibs_metamodel_celltypes_v661')
nucleus_functional_area_assignment= client.materialize.query_table('nucleus_functional_area_assignment')
functional_properties_v3_bcm = client.materialize.tables.functional_properties_v3_bcm().query()
proofreading_status_and_strategy = client.materialize.query_table('proofreading_status_and_strategy')

The `client.materialize.tables` interface is experimental and might experience breaking changes before the feature is stabilized.


In [10]:
aibs_metamodel_celltypes_v661.head()

,id,created,valid,target_id,classification_system,cell_type,id_ref,created_ref,valid_ref,volume,pt_supervoxel_id,pt_root_id,pt_position,bb_start_position,bb_end_position
0,36916,2023-12-19 22:47:18.659864+00:00,t,336365,excitatory_neuron,5P-IT,336365,2020-09-28 22:42:48.966292+00:00,t,272.488202,93606511657924288,864691136274724621,"[209760, 180832, 27076]","[nan, nan, nan]","[nan, nan, nan]"
1,1070,2023-12-19 22:38:00.472115+00:00,t,110648,excitatory_neuron,23P,110648,2020-09-28 22:45:09.650639+00:00,t,328.533443,79385153184885329,864691135489403194,"[106448, 129632, 25410]","[nan, nan, nan]","[nan, nan, nan]"
2,1099,2023-12-19 22:38:00.898837+00:00,t,112071,excitatory_neuron,23P,112071,2020-09-28 22:43:34.088785+00:00,t,272.929423,79035988248401958,864691136147292311,"[103696, 149472, 15583]","[nan, nan, nan]","[nan, nan, nan]"
3,13259,2023-12-19 22:41:14.417986+00:00,t,197927,nonneuron,oligo,197927,2020-09-28 22:43:10.652649+00:00,t,91.308851,84529699506051734,864691136050858227,"[143600, 186192, 26471]","[nan, nan, nan]","[nan, nan, nan]"
4,13271,2023-12-19 22:41:14.685474+00:00,t,198087,nonneuron,astrocyte,198087,2020-09-28 22:41:36.677186+00:00,t,161.744978,83756261929388963,864691135809440972,"[137952, 190944, 27361]","[nan, nan, nan]","[nan, nan, nan]"


In [4]:
# ======== Saving table ============
path_save_ann = "/data/data_raw/annotation_tables/"

proofreading_status_and_strategy_fname = "proofreading_status_and_strategy"
coregistration_manual_fname = "coregistration_manual_v4"
nucleus_detection_fname = "nucleus_detection_v0"
aibs_metamodel_celltypes_fname = "aibs_metamodel_celltypes_v661"
nucleus_functional_area_fname = "nucleus_functional_area_assignment"
functional_properties_fname = "functional_properties_v3_bcm"

proofreading_status_and_strategy.to_pickle(path_project + path_save_ann + proofreading_status_and_strategy_fname)
coregistration_manual_v4.to_pickle(path_project + path_save_ann + coregistration_manual_fname)
nucleus_detection_v0.to_pickle(path_project + path_save_ann + nucleus_detection_fname)
aibs_metamodel_celltypes_v661.to_pickle(path_project + path_save_ann + aibs_metamodel_celltypes_fname)
nucleus_functional_area_assignment.to_pickle(path_project + path_save_ann + nucleus_functional_area_fname)
functional_properties_v3_bcm.to_pickle(path_project + path_save_ann + functional_properties_fname)

In [13]:
# ==== To selected dendrites/axons proofread ======
# dend_extended_df = client.materialize.query_table('proofreading_status_and_strategy',
#                                                   filter_in_dict={"strategy_dendrite": ["dendrite_extended"]})
# axon_true = client.materialize.query_table('proofreading_status_and_strategy',
#                                                 filter_in_dict={"strategy_axon": ["axon_interareal","axon_fully_extended","axon_partially_extended","axon_column_truncated"]})

## Filtered neurons

Save selected set of neurons

### Proofread neurons

Here we save set of proofread neurons details refer the code

In [11]:
# get clean and extended version of dendrites proofreading
def remove_unhashable_columns(df):
    return df[[col for col in df.columns if df[col].apply(lambda x: not isinstance(x, (list, np.ndarray))).all()]]
    
# Query tables
dend_extended_df = client.materialize.query_table('proofreading_status_and_strategy',
                                                  filter_in_dict={"strategy_dendrite": ["dendrite_extended"]})
dend_clean_df = client.materialize.query_table('proofreading_status_and_strategy', 
                                               filter_in_dict={"strategy_dendrite": ["dendrite_clean"]})
# remove them
dend_clean_df_cleaned = remove_unhashable_columns(dend_clean_df)
dend_extended_df_cleaned = remove_unhashable_columns(dend_extended_df)

# Combine
dend_combined_df = pd.concat([dend_clean_df_cleaned, dend_extended_df_cleaned]).drop_duplicates().reset_index(drop=True)

# List of neurons
list_pt_root_id = pd.merge(dend_combined_df[['pt_root_id']], coregistration_manual_v4[['pt_root_id']],how='inner',on=['pt_root_id'])

# add single cell type (23P or 4P)
mask = aibs_metamodel_celltypes_v661['cell_type'] == '23P'
list_pt_root_id = pd.merge(list_pt_root_id, aibs_metamodel_celltypes_v661[mask][['pt_root_id']], how='inner', on=['pt_root_id'])

# # add V5 cell type
# mask = aibs_metamodel_celltypes_v661['cell_type'].isin(['5P-IT', '5P-ET', '5P-NP'])
# list_pt_root_id = pd.merge(list_pt_root_id, aibs_metamodel_celltypes_v661[mask][['pt_root_id']], how='inner', on=['pt_root_id'])

mask = nucleus_functional_area_assignment['tag'] == 'V1'
list_pt_root_id = pd.merge(list_pt_root_id, nucleus_functional_area_assignment[mask][['pt_root_id']], how='inner', on=['pt_root_id'])

# Check its functional properties are known and the neuron has a good selectivity (large gOSI)
mask = functional_properties_v3_bcm['gOSI'] != None
list_pt_root_id= pd.merge(list_pt_root_id['pt_root_id'], functional_properties_v3_bcm[mask]['pt_root_id'], how='inner', on=['pt_root_id'])

list_uniq_root_id = list_pt_root_id.pt_root_id.unique()
list_uniq_root_id = pd.DataFrame({'pt_root_id': list_uniq_root_id})

print(f"Total neurons selected: {len(list_pt_root_id)}")
print(f"Number of unique neurons: {len(list_uniq_root_id)} \n")
list_uniq_root_id

Total neurons selected: 443
Number of unique neurons: 224 



,pt_root_id
0,864691136663700958
1,864691136330407914
2,864691135953669667
3,864691135492023271
4,864691136090670903
...,...
219,864691135417045946
220,864691135778419901
221,864691135782048080
222,864691135778467517


In [17]:
# === Save list ===
proofread_23P = list_uniq_root_id['pt_root_id'].to_list()
np.savetxt(path_project + '/data/proofread_23P.csv', proofread_23P, delimiter=',')

### Including non proofread neurons

To expand our neurons, we use all the nonproofread neurons as well - this includes proofread neurons

In [26]:
# === List of 23 layer V1 neurons without proofreading ===
mask = nucleus_detection_v0['pt_root_id'] != 0
list_id = nucleus_detection_v0[mask][['id']]

mask = aibs_metamodel_celltypes_v661['cell_type'] == '23P'
list_id = pd.merge(list_id, aibs_metamodel_celltypes_v661[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

mask = nucleus_functional_area_assignment['tag'] == 'V1'
list_id = pd.merge(list_id, nucleus_functional_area_assignment[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

list_id = pd.merge(list_id, coregistration_manual_v4[['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

# ---- Drop duplicates ----
list_id = list_id.drop_duplicates()

# Final list of pt_root_id - since list_id is a id, and we want to bring pt_root_id
list_pt_root_id = nucleus_detection_v0[nucleus_detection_v0['id'].isin(list_id['id'])][['pt_root_id']]
list_pt_root_id

,pt_root_id
51,864691135275621605
57,864691135738006641
73,864691135463256477
75,864691135748985641
174,864691135786545860
...,...
143346,864691135492080871
143398,864691135976467523
143407,864691135162777133
143493,864691135279141793


In [31]:
# === Save list ===
nonproofread_23P = list_pt_root_id['pt_root_id'].to_list()
np.savetxt(path_project + '/data/neuron_list/nonproofread_23P.csv', nonproofread_23P, delimiter=',')

## Synapses

Query pre and post synapses\
Have to select list of neurons first

### Filter neurons

In [10]:
# === List of 23 layer V1 neurons without proofreading ===
mask = nucleus_detection_v0['pt_root_id'] != 0
list_id = nucleus_detection_v0[mask][['id']]

mask = aibs_metamodel_celltypes_v661['cell_type'] == '23P'
list_id = pd.merge(list_id, aibs_metamodel_celltypes_v661[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

mask = nucleus_functional_area_assignment['tag'] == 'V1'
list_id = pd.merge(list_id, nucleus_functional_area_assignment[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

list_id = pd.merge(list_id, coregistration_manual_v4[['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

In [11]:
list_id

,id
0,190151
1,189115
2,153275
3,153085
4,391162
...,...
6293,360127
6294,421011
6295,421011
6296,328981


In [12]:
# Drop duplicates
list_id = list_id.drop_duplicates()

In [16]:
# Final list of pt_root_id - since list_id is a id, and we want to bring pt_root_id
list_pt_root_id = nucleus_detection_v0[nucleus_detection_v0['id'].isin(list_id['id'])][['pt_root_id']]
list_pt_root_id

,pt_root_id
51,864691135275621605
57,864691135738006641
73,864691135463256477
75,864691135748985641
174,864691135786545860
...,...
143346,864691135492080871
143398,864691135976467523
143407,864691135162777133
143493,864691135279141793


### pre-synapses

Download postsynaptic neuron's presynapse data that includes synapse_id & location

In [ ]:
# ============= Before you begin, exclude neurons that already have presynaptic data =====
# # Get list of already existing files
# cell_type = "23P_V1_old/"
# dir_name = "/media/DATA1/CK/python_script/MicronsBinder_03/results/skeleton/pre_synapse_data/"
# file_name = "*_pre_synapse_df.pkl"
# filenames = glob.glob(dir_name + cell_type + file_name)

# neuron_exists = []
# for fname in filenames:
#     # extract numeric ID from filename
#     numeric_id = int(os.path.basename(fname).split("_")[0])
#     neuron_exists.append(numeric_id)
    
# print(f"Already existing neurons = {len(neuron_exists)}")

In [ ]:
# path set
path_pre = "/data/synapse/"

# Apply filtered list of neurons
filtered_list_pt_root_id = list_pt_root_id[~list_pt_root_id['pt_root_id'].isin(neuron_exists)]

# Split for batch processing
batches = np.array_split(filtered_list_pt_root_id, 20)

# Change number from failed batch
for num in range(19,21):
    print(f"Processing batch {num}/{len(batches)} …")
    
    for neuron in batches[num]['pt_root_id']:
        pre_synapse_df = client.materialize.synapse_query(post_ids = neuron)
    
        fname = f"{neuron}_pre_synapse_df.pkl"
        with open(path_project + path_pre + fname, 'wb') as f:
            pickle.dump(pre_synapse_df, f)
            
    print(f"{num} batch successfully saved\n\n")

### post-synapses

Download postsynaptic neuron's presynapse data that includes synapse_id & location

In [ ]:
# Path set
path_post = "/data/synapse/"

# Split for batch processing
batches = np.array_split(list_uniq_root_id, 20)

# Change number from failed batch
#for num in range(len(batches)):
for num in range(1,20):
    print(f"processing batch {num}/{len(batches)} ...")

    for neuron in batches[num]:
        post_synapse_df = client.materialize.synapse_query(pre_ids = neuron)

        fname = f"{neuron}_post_synapse_df.pkl"
        with open(path_project + path_post + fname, 'wb') as f:
            pickle.dump(post_synapse_df, f)
            
    print(f"{num} batch successfully saved\n")

## Skeletons

Download swc skeleton data

### Filter neurons

In [17]:
# === List of 23 layer V1 neurons without proofreading ===
mask = nucleus_detection_v0['pt_root_id'] != 0
list_id = nucleus_detection_v0[mask][['id']]

mask = aibs_metamodel_celltypes_v661['cell_type'] == '23P'
list_id = pd.merge(list_id, aibs_metamodel_celltypes_v661[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

mask = nucleus_functional_area_assignment['tag'] == 'V1'
list_id = pd.merge(list_id, nucleus_functional_area_assignment[mask][['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

list_id = pd.merge(list_id, coregistration_manual_v4[['target_id']], 
                           how='inner', left_on = 'id', right_on = 'target_id')
list_id = list_id[['id']]

In [ ]:
""" Prepare list of batches """

# Convert list_uniq_root_id into list for Batch processing
list_uniq_root_id_list = list_uniq_root_id['pt_root_id'].tolist()

# Split into chunks of 15 neurons each
# This will seperate data into 15 neurons each and save them in neuron_chunks altogether
chunk_size = 15
neuron_chunks = [list_uniq_root_id_list[i:i + chunk_size] for i in range(0, len(list_uniq_root_id_list), chunk_size)]

# create labels for each chunk ex) list_uniq_neurons_01, list_uniq_neurons_02, ...  
chunk_labels = [f"list_uniq_neurons_{str(i+1).zfill(2)}" for i in range(len(neuron_chunks))]

In [ ]:
# Batch processing test
for chunk_index, chunk in enumerate(neuron_chunks):
    chunk_label = chunk_labels[chunk_index]
    
    print(f"Processing {chunk_label}...")

    for root_id in chunk:
        print(root_id)

In [ ]:
""" Saving raw skeleton data - not preprocessed"""
# Saving raw skeleton data
dir_save_raw = "/media/DATA1/CK/python_script/MicronsBinder_03/results/skeleton/skeleton_swc_data/skeleton_swc_raw/"

start_chunk_index = 0  # Change this to the failed chunk index (e.g., 1 for chunk 2)

for chunk_index in range(start_chunk_index, len(neuron_chunks)):  # Start from failed chunk
    chunk_label = chunk_labels[chunk_index]
    
    print(f"Processing {chunk_label} (Chunk {chunk_index+1})...")

    for root_id in neuron_chunks[chunk_index]:
        try:
            # Load skeleton
            sk_swc = client.skeleton.get_skeleton(root_id, output_format='swc', skeleton_version=2)
            
            # Save skeleton
            file_name = f"{root_id}_sk_swc_raw.csv"
            sk_swc.to_csv(dir_save_raw + file_name)
        
        except Exception as e:
            print(f"Error processing neuron {root_id}: {e}")

In [2]:
# Caveclient setup
import caveclient
client = caveclient.CAVEclient('minnie65_public')
client.version = 1300
sk_swc = client.skeleton.get_skeleton(864691135614771147, output_format='swc', skeleton_version=2)
sk_swc

,id,type,x,y,z,radius,parent
0,0,1,723.904,575.552,886.48,5.772,-1
1,1,3,726.224,585.152,884.56,0.258,0
2,2,2,726.080,587.608,885.20,0.273,1
3,3,2,726.408,589.512,886.08,0.273,2
4,4,2,726.712,591.544,887.32,0.273,3
...,...,...,...,...,...,...,...
6215,6215,3,752.760,587.064,879.68,0.147,6214
6216,6216,3,754.960,587.304,879.40,0.147,6215
6217,6217,3,757.336,586.736,879.56,0.147,6216
6218,6218,3,758.336,586.560,879.04,0.147,6217
